# **Reservoir Capacity Forecasting — ETL Pipeline**

## **Context**
The Instituto Nacional de Meteorología commissioned an analysis of the Madrid 
reservoir system to predict drought risk and regulate water consumption proactively.

Data source: Canal de Isabel II — 14 reservoirs, monthly capacity measurements 
(hm³), 1998–2021.

## **1. Load & Combine**

14 CSV files, one per reservoir. Each file has:
- BOM-prefixed `anio` column (`ï»¿anio`) — handled via `utf-8-sig` encoding
- European decimal commas (`326,784` → `326.784`)
- Year only appears in January rows — remaining months left empty
- 13 trailing empty columns

All loading and combining logic is encapsulated in `src/etl.load_raw_csvs()`.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from src.etl import load_raw_csvs, clean_dataframe, build_pivot

PROJECT_ROOT = Path.cwd().parent
DATA_RAW     = PROJECT_ROOT / "data" / "raw"
DATA_PROC    = PROJECT_ROOT / "data" / "processed"
DATA_PROC.mkdir(parents=True, exist_ok=True)

print(f"Raw CSVs: {len(list(DATA_RAW.glob('*.csv')))}")

Raw CSVs: 14


In [2]:
df_raw = load_raw_csvs(DATA_RAW)
print(f"Shape: {df_raw.shape}")
df_raw.head()

Shape: (4076, 4)


,year,month,capacity_hm3,reservoir
0,1998.0,enero,"23,891",RioCofio_LaAcena
1,NaN,febrero,"23,816",RioCofio_LaAcena
2,NaN,marzo,"23,245",RioCofio_LaAcena
3,NaN,abril,"23,35",RioCofio_LaAcena
4,NaN,mayo,"23,678",RioCofio_LaAcena


## **2. Clean**

Key decisions:

**Year reconstruction** — `RioLozoya_Riosequillo` has all year values missing. 
Years are reconstructed from the known sequence 1998–2021 (288 rows, 12 per year).
All other reservoirs use forward-fill per reservoir group after BOM fix.

**`RioLosMorales_LosMorales` excluded** — data ends in 2018, 3 years short of 
the rest. Including it would require imputation over 36 months, introducing more 
noise than signal.

**Truncation to March 2021** — all remaining reservoirs end in March 2021. 
Rows for April–December 2021 have no capacity data and are dropped.

In [3]:
df_clean = clean_dataframe(df_raw)
print(f"Shape: {df_clean.shape}")
print(f"Nulls: {df_clean['capacity_hm3'].isna().sum()}")
print(f"Reservoirs: {sorted(df_clean['reservoir'].unique())}")
df_clean.head()

Shape: (3627, 4)
Nulls: 0
Reservoirs: ['RioCofio_LaAcena', 'RioGuadalix_Pedrezuela', 'RioGuadarrama-Aulencia_LaJorosa', 'RioGuadarrama-Aulencia_Navalmedio', 'RioGuadarrama-Aulencia_Valmayor', 'RioJarama_ElVado', 'RioLozoya_ElAtazar', 'RioLozoya_ElVillar', 'RioLozoya_LaPinilla', 'RioLozoya_PuentesViejas', 'RioLozoya_Riosequillo', 'RioManzanares_Navacerrada', 'RioManzanares_Santillana']


,year,month,capacity_hm3,reservoir
0,1998,enero,23.891,RioCofio_LaAcena
1,1998,febrero,23.816,RioCofio_LaAcena
2,1998,marzo,23.245,RioCofio_LaAcena
3,1998,abril,23.350,RioCofio_LaAcena
4,1998,mayo,23.678,RioCofio_LaAcena


## **3. Pivot**

Long-format DataFrame pivoted to wide-format:
- One row per month (279 rows: Jan 1998 – Mar 2021)
- One column per reservoir (13 columns)
- `total_hm3` = sum of all reservoirs per month
- `ds` = datetime index for time series work

In [4]:
df_pivot = build_pivot(df_clean)
print(f"Shape: {df_pivot.shape}")
print(f"Date range: {df_pivot['ds'].min()} → {df_pivot['ds'].max()}")
df_pivot.head()

Shape: (279, 17)
Date range: 1998-01-01 00:00:00 → 2021-03-01 00:00:00


,ds,year,month,total_hm3,RioCofio_LaAcena,RioGuadalix_Pedrezuela,RioGuadarrama-Aulencia_LaJorosa,RioGuadarrama-Aulencia_Navalmedio,RioGuadarrama-Aulencia_Valmayor,RioJarama_ElVado,RioLozoya_ElAtazar,RioLozoya_ElVillar,RioLozoya_LaPinilla,RioLozoya_PuentesViejas,RioLozoya_Riosequillo,RioManzanares_Navacerrada,RioManzanares_Santillana
0,1998-01-01,1998,enero,788.352,23.891,38.413,6.572,0.182,118.990,45.882,326.784,22.741,30.331,41.769,45.782,10.062,76.953
1,1998-02-01,1998,febrero,790.650,23.816,38.104,6.462,0.617,119.578,46.043,327.590,21.550,30.254,41.442,46.640,10.114,78.440
2,1998-03-01,1998,marzo,785.899,23.245,33.449,6.848,0.500,119.063,39.928,336.267,21.670,30.020,40.911,45.965,9.966,78.067
3,1998-04-01,1998,abril,810.614,23.350,27.848,6.747,0.639,119.578,42.443,353.267,21.644,31.674,44.173,46.209,10.326,82.716
4,1998-05-01,1998,mayo,864.033,23.678,23.009,7.041,0.625,120.833,49.165,391.448,21.925,35.518,50.395,43.954,10.989,85.453


## **4. Validation & Export**

- 0 nulls in `total_hm3`
- Monotonically increasing datetime index
- Exported to `data/processed/reservoirs_pivot.csv` and `reservoirs_long.csv`

In [5]:
assert df_pivot["total_hm3"].isna().sum() == 0
assert df_pivot["ds"].is_monotonic_increasing

df_pivot.to_csv(DATA_PROC / "reservoirs_pivot.csv", index=False)
df_clean.to_csv(DATA_PROC / "reservoirs_long.csv", index=False)

print(f"Exported reservoirs_pivot.csv — {df_pivot.shape}")
print(f"Exported reservoirs_long.csv  — {df_clean.shape}")

Exported reservoirs_pivot.csv — (279, 17)
Exported reservoirs_long.csv  — (3627, 4)
